# B3b Defense – 03 Feature Engineering

**Objective:** Build `defense_feature_matrix.csv` from `macro_features_defense.csv`.
Target variable: ADEFNO (absolute level). Features include:
- Lag features on ADEFNO_diff, IPB52300S_diff (lags 1–6)
- Cross-sector lags on FDEFX_diff (lags 1–3)
- Autoregressive lags on ADEFNO absolute (lags 1–3)
- Rolling window statistics on ADEFNO (3m and 6m mean/std)
- Calendar features (month, quarter, year, is_q4)

In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
DATA_PROCESSED = '../data/processed/'

# Load macro_features_defense.csv and parse date as DatetimeIndex
input_path = os.path.join(DATA_PROCESSED, 'macro_features_defense.csv')
df = pd.read_csv(input_path, parse_dates=['date'])
df.set_index('date', inplace=True)
df.sort_index(inplace=True)

print(f'Loaded: {input_path}')
print(f'Shape: {df.shape}')
print(f'Date range: {df.index.min().date()} to {df.index.max().date()}')
df.head()

Loaded: ../data/processed/macro_features_defense.csv
Shape: (311, 6)
Date range: 2000-02-01 to 2025-12-01


,ADEFNO,ADEFNO_diff,IPB52300S,IPB52300S_diff,FDEFX,FDEFX_diff
date,,,,,,
2000-02-01,5162,-1997.0,69.3458,-1.6863,383.028,0.000
2000-03-01,3773,-1389.0,68.6844,-0.6614,383.028,0.000
2000-04-01,3568,-205.0,67.3689,-1.3155,399.592,16.564
2000-05-01,5023,1455.0,66.5616,-0.8073,399.592,0.000
2000-06-01,25893,20870.0,67.1028,0.5412,399.592,0.000


In [3]:
# Target: ADEFNO absolute (not differenced).
# Rationale: absolute values preserve interpretability — forecast outputs can be
# read directly in USD millions without cumulative reconstruction from diffs.
target = df['ADEFNO'].copy()

# Initialize feature DataFrame starting from the raw data
feat = df.copy()

In [4]:
# Lag features for ADEFNO_diff (lags 1–6)
for lag in range(1, 7):
    feat[f'ADEFNO_diff_lag_{lag}'] = feat['ADEFNO_diff'].shift(lag)

print('ADEFNO_diff lags 1–6 created.')

ADEFNO_diff lags 1–6 created.


In [5]:
# Lag features for IPB52300S_diff (lags 1–6)
for lag in range(1, 7):
    feat[f'IPB52300S_diff_lag_{lag}'] = feat['IPB52300S_diff'].shift(lag)

print('IPB52300S_diff lags 1–6 created.')

IPB52300S_diff lags 1–6 created.


In [6]:
# Cross-sector lag features: FDEFX_diff (lags 1–3)
# FDEFX measures realized federal defense expenditures — a complementary perspective
# to ADEFNO (ordering intent) and IPB52300S (production capacity).
for lag in range(1, 4):
    feat[f'FDEFX_diff_lag_{lag}'] = feat['FDEFX_diff'].shift(lag)

print('FDEFX_diff lags 1–3 created.')

FDEFX_diff lags 1–3 created.


In [7]:
# Autoregressive lags on ADEFNO absolute level (lags 1–3)
for lag in range(1, 4):
    feat[f'adefno_lag_{lag}'] = feat['ADEFNO'].shift(lag)

print('ADEFNO absolute AR lags 1–3 created.')

ADEFNO absolute AR lags 1–3 created.


In [8]:
# Rolling window features on ADEFNO absolute level
# shift(1) is applied before rolling to prevent data leakage (no look-ahead)
adefno_shifted = feat['ADEFNO'].shift(1)

feat['adefno_rolling_3m_mean'] = adefno_shifted.rolling(window=3).mean()
feat['adefno_rolling_3m_std']  = adefno_shifted.rolling(window=3).std()
feat['adefno_rolling_6m_mean'] = adefno_shifted.rolling(window=6).mean()
feat['adefno_rolling_6m_std']  = adefno_shifted.rolling(window=6).std()

print('Rolling window features (3m, 6m mean/std) created with shift=1 to avoid leakage.')

Rolling window features (3m, 6m mean/std) created with shift=1 to avoid leakage.


In [9]:
# Calendar features derived from the DatetimeIndex
feat['month']   = feat.index.month
feat['quarter'] = feat.index.quarter
feat['year']    = feat.index.year
feat['is_q4']   = (feat['quarter'] == 4).astype(int)

print('Calendar features created: month, quarter, year, is_q4.')

Calendar features created: month, quarter, year, is_q4.


In [10]:
# Drop NaN rows introduced by lag and rolling operations
shape_before = feat.shape
feat.dropna(inplace=True)
shape_after = feat.shape

print(f'Shape before dropna: {shape_before}')
print(f'Shape after  dropna: {shape_after}')
print(f'Rows dropped: {shape_before[0] - shape_after[0]}')

Shape before dropna: (311, 32)
Shape after  dropna: (305, 32)
Rows dropped: 6


In [11]:
# Save defense_feature_matrix.csv to DATA_PROCESSED
out_path = os.path.join(DATA_PROCESSED, 'defense_feature_matrix.csv')
feat.to_csv(out_path)
print(f'Saved: {out_path}')
print()
print('head(3):')
print(feat.head(3).to_string())
print()
print('dtypes:')
print(feat.dtypes)

Saved: ../data/processed/defense_feature_matrix.csv

head(3):
            ADEFNO  ADEFNO_diff  IPB52300S  IPB52300S_diff    FDEFX  FDEFX_diff  ADEFNO_diff_lag_1  ADEFNO_diff_lag_2  ADEFNO_diff_lag_3  ADEFNO_diff_lag_4  ADEFNO_diff_lag_5  ADEFNO_diff_lag_6  IPB52300S_diff_lag_1  IPB52300S_diff_lag_2  IPB52300S_diff_lag_3  IPB52300S_diff_lag_4  IPB52300S_diff_lag_5  IPB52300S_diff_lag_6  FDEFX_diff_lag_1  FDEFX_diff_lag_2  FDEFX_diff_lag_3  adefno_lag_1  adefno_lag_2  adefno_lag_3  adefno_rolling_3m_mean  adefno_rolling_3m_std  adefno_rolling_6m_mean  adefno_rolling_6m_std  month  quarter  year  is_q4
date                                                                                                                                                                                                                                                                                                                                                                                                     